# Face Classify Studio

这个 Notebook 是项目的主源码入口，覆盖 7 分类人脸识别的完整流程：数据加载、迁移学习训练、验证测试、单图推理、测试人员反馈微调，以及桌面端启动。

类别包括：`Black`、`East Asian`、`Indian`、`Latino_Hispanic`、`Middle Eastern`、`Southeast Asian`、`White`。

## 1. 安装依赖

首次运行时执行下面这一格。已经安装过依赖后可以跳过。

In [ ]:
# %pip install -r requirements.txt

## 2. 导入库与基础配置

In [ ]:
from pathlib import Path
import copy
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision import datasets, models, transforms

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "model"
CHECKPOINT_PATH = MODEL_DIR / "best_resnet18_faces.pth"
FEEDBACK_DIR = PROJECT_ROOT / "feedback" / "images"

CLASS_NAMES = [
    "Black",
    "East Asian",
    "Indian",
    "Latino_Hispanic",
    "Middle Eastern",
    "Southeast Asian",
    "White",
]
IMAGE_SIZE = 224
BATCH_SIZE = 8
SEED = 42

def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
device = get_device()
device

## 3. 数据加载

数据目录需要符合 PyTorch `ImageFolder` 格式：

```text
data/train/<class_name>/*.jpg
data/val/<class_name>/*.jpg
data/test/<class_name>/*.jpg
```

训练集使用随机增强，验证集和测试集只做标准化。

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ColorJitter(brightness=0.12, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def build_dataloaders(data_dir=DATA_DIR, batch_size=BATCH_SIZE):
    train_dataset = datasets.ImageFolder(data_dir / "train", transform=train_transform)
    val_dataset = datasets.ImageFolder(data_dir / "val", transform=eval_transform)
    test_dataset = datasets.ImageFolder(data_dir / "test", transform=eval_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader

# train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = build_dataloaders()
# train_dataset.classes, len(train_dataset), len(val_dataset), len(test_dataset)

## 4. 构建 ResNet18 迁移学习模型

最后一层全连接层替换为 7 类 logits。训练时使用 `CrossEntropyLoss`，不需要提前加 Softmax。

In [ ]:
def build_model(num_classes=7, pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def freeze_backbone(model):
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("fc.")

def unfreeze_for_finetune(model, mode="layer4"):
    for param in model.parameters():
        param.requires_grad = False
    prefixes = ["fc.", "layer4."] if mode == "layer4" else ["fc.", "layer3.", "layer4."]
    for name, param in model.named_parameters():
        if any(name.startswith(prefix) for prefix in prefixes):
            param.requires_grad = True

model = build_model(num_classes=len(CLASS_NAMES)).to(device)
model.fc

## 5. 训练与评估函数

In [ ]:
def run_one_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return total_loss / total, correct / total

def train_phase(model, train_loader, val_loader, epochs, lr, weight_decay=1e-4, patience=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=weight_decay,
    )
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0
    stale_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_one_epoch(model, train_loader, criterion, device, optimizer)
        val_loss, val_acc = run_one_epoch(model, val_loader, criterion, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })
        print(f"epoch {epoch:02d}/{epochs} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    return history, best_val_acc

def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            y_true.extend(labels.numpy().tolist())
    return y_true, y_pred

## 6. 两阶段训练

第一阶段冻结骨干，只训练分类头；第二阶段解冻 `layer4` 或 `layer3 + layer4` 做小学习率微调。

In [ ]:
# train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = build_dataloaders()
# model = build_model(num_classes=len(train_dataset.classes)).to(device)
#
# freeze_backbone(model)
# history_head, best_head_acc = train_phase(model, train_loader, val_loader, epochs=15, lr=1e-3)
#
# unfreeze_for_finetune(model, mode="layer3_layer4")
# history_ft, best_ft_acc = train_phase(model, train_loader, val_loader, epochs=12, lr=1e-5)
#
# MODEL_DIR.mkdir(parents=True, exist_ok=True)
# torch.save({
#     "model_state_dict": model.state_dict(),
#     "class_names": train_dataset.classes,
#     "image_size": IMAGE_SIZE,
#     "best_val_acc": best_ft_acc,
# }, CHECKPOINT_PATH)
# print(f"Saved checkpoint: {CHECKPOINT_PATH}")

## 7. 测试集评估

In [ ]:
# y_true, y_pred = collect_predictions(model, test_loader)
# report = classification_report(
#     y_true,
#     y_pred,
#     labels=list(range(len(train_dataset.classes))),
#     target_names=train_dataset.classes,
#     digits=4,
#     zero_division=0,
# )
# print(report)
# print(confusion_matrix(y_true, y_pred, labels=list(range(len(train_dataset.classes)))))

## 8. 加载已训练模型并做单图推理

In [ ]:
def load_checkpoint(checkpoint_path=CHECKPOINT_PATH):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    class_names = checkpoint["class_names"]
    model = build_model(num_classes=len(class_names), pretrained=False).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, class_names

def predict_image(image_path, checkpoint_path=CHECKPOINT_PATH):
    model, class_names = load_checkpoint(checkpoint_path)
    image = Image.open(image_path).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    results = sorted(zip(class_names, probs), key=lambda x: x[1], reverse=True)
    return [{"class_name": name, "probability": float(prob)} for name, prob in results]

# results = predict_image("path/to/image.jpg")
# results[:3]

## 9. 反馈样本微调

桌面端保存的错误反馈位于 `feedback/images/<正确标签>/`。反馈微调只解冻 `layer4 + fc`，使用很小学习率，避免单次反馈破坏原模型。

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class FeedbackDataset(Dataset):
    def __init__(self, feedback_dir, class_names, transform=None, repeat=8):
        self.samples = []
        self.transform = transform
        self.repeat = max(1, repeat)
        class_to_idx = {name: idx for idx, name in enumerate(class_names)}
        for class_name in class_names:
            class_dir = Path(feedback_dir) / class_name
            if not class_dir.exists():
                continue
            for path in sorted(class_dir.iterdir()):
                if path.suffix.lower() in IMAGE_SUFFIXES:
                    self.samples.append((path, class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples) * self.repeat

    def __getitem__(self, index):
        path, label = self.samples[index % len(self.samples)]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def feedback_finetune(epochs=5, lr=5e-6, repeat=8):
    train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = build_dataloaders()
    feedback_dataset = FeedbackDataset(FEEDBACK_DIR, train_dataset.classes, train_transform, repeat=repeat)
    if len(feedback_dataset.samples) == 0:
        raise RuntimeError("No feedback samples found. Use the desktop app to save feedback first.")

    combined_train = ConcatDataset([train_dataset, feedback_dataset])
    combined_loader = DataLoader(combined_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    model, class_names = load_checkpoint(CHECKPOINT_PATH)
    unfreeze_for_finetune(model, mode="layer4")
    history, best_val_acc = train_phase(model, combined_loader, val_loader, epochs=epochs, lr=lr)

    output_path = MODEL_DIR / "best_resnet18_faces_feedback.pth"
    torch.save({
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
        "feedback_training": {
            "feedback_images": len(feedback_dataset.samples),
            "repeat": repeat,
            "epochs": epochs,
            "lr": lr,
            "best_val_acc": best_val_acc,
        },
    }, output_path)
    print(f"Saved feedback checkpoint: {output_path}")
    return output_path, history

# feedback_finetune()

## 10. 启动桌面端

Notebook 负责展示和复现实验流程；桌面端仍可以作为本地推理工具启动。

In [ ]:
# %run code/desktop_app.ipynb